# 学习信号，忽略噪声：正则化和批处理

这一份 Notebook 用一个小型回归任务观察：训练误差和验证误差为什么会分开，以及正则化如何改变这种现象。

实验顺序是：准备带噪声的数据 → 切分训练集和验证集 → 训练模型 → 比较两种误差 → 加入正则化。

In [1]:
import numpy as np

rng = np.random.default_rng(0)

X = np.linspace(-1, 1, 30).reshape(-1, 1)
clean_targets = np.sin(3 * X[:, 0])
targets = clean_targets + rng.normal(0, 0.15, size=len(X))

indices = rng.permutation(len(X))
train_indices = indices[:20]
validation_indices = indices[20:]

X_train = X[train_indices]
y_train = targets[train_indices, None]
X_validation = X[validation_indices]
y_validation = targets[validation_indices, None]

print('X_train.shape =', X_train.shape)
print('y_train.shape =', y_train.shape)
print('X_validation.shape =', X_validation.shape)
print('y_validation.shape =', y_validation.shape)


X_train.shape = (20, 1)
y_train.shape = (20, 1)
X_validation.shape = (10, 1)
y_validation.shape = (10, 1)


## 先确认数据角色

训练集用于更新参数，验证集只用于观察泛化能力：

```text
训练集 → 计算梯度、更新权重
验证集 → 只计算误差，不更新权重
```

这里的目标是带噪声的 `sin(3x)`。模型应该学习总体趋势，而不是死记每个噪声点。

## 先训练一个容量较大的模型

我们暂时不加入任何正则化，让隐藏层拥有 32 个节点。训练过程中同时记录：

```text
train_loss：用于更新参数
validation_loss：只用于观察泛化
```

如果模型开始记住训练集中的噪声，训练误差会继续下降，而验证误差可能不再下降。

In [ ]:
from layers import (
    linear_forward,
    linear_backward,
    relu_forward,
    relu_backward,
    mse_loss,
    mse_backward,
)

model_rng = np.random.default_rng(1)
W1 = model_rng.normal(0, 0.5, size=(1, 32))
b1 = np.full(32, 0.1)
W2 = model_rng.normal(0, 0.5, size=(32, 1))
b2 = np.zeros(1)
alpha = 0.02

for epoch in range(5000):
    # 训练集前向传播
    z1, cache1 = linear_forward(X_train, W1, b1)
    hidden, cache_relu = relu_forward(z1)
    predictions, cache2 = linear_forward(hidden, W2, b2)

    train_loss = mse_loss(predictions, y_train)

    # 训练集反向传播与参数更新
    dz2 = mse_backward(predictions, y_train)
    d_hidden, dW2, db2 = linear_backward(dz2, cache2)
    dz1 = relu_backward(d_hidden, cache_relu)
    dX, dW1, db1 = linear_backward(dz1, cache1)

    W1 -= alpha * dW1
    b1 -= alpha * db1
    W2 -= alpha * dW2
    b2 -= alpha * db2

    # 验证集只前向，不更新参数
    validation_z1, _ = linear_forward(X_validation, W1, b1)
    validation_hidden, _ = relu_forward(validation_z1)
    validation_predictions, _ = linear_forward(validation_hidden, W2, b2)
    validation_loss = mse_loss(validation_predictions, y_validation)

    if epoch % 1000 == 0:
        print(
            'epoch =', epoch,
            'train_loss =', train_loss,
            'validation_loss =', validation_loss
        )


## 观察重点

先不要急着判断模型好不好，观察两条误差的相对变化：

- 如果两者都下降，模型还在学习总体规律；
- 如果训练误差继续下降、验证误差停止下降甚至上升，开始出现过拟合；
- 验证误差最低的 epoch，通常比最后一个 epoch 更值得保留。

下一步我们会保存误差曲线，并加入提前停止，让模型停在泛化能力最好的位置。

## 保存最佳模型：提前停止

提前停止的逻辑很简单：验证误差改善时保存参数；连续很多轮没有改善，就停止训练。

```text
validation_loss 变小 → 保存 W、b
validation_loss 不再变小 → 累计等待次数
等待超过 patience → 停止
```

注意保存参数时要使用 `.copy()`，否则保存的只是指向同一块数组的引用。

In [ ]:
model_rng = np.random.default_rng(1)
W1 = model_rng.normal(0, 0.5, size=(1, 32))
b1 = np.full(32, 0.1)
W2 = model_rng.normal(0, 0.5, size=(32, 1))
b2 = np.zeros(1)

alpha = 0.02
patience = 500
wait = 0
best_validation_loss = np.inf
best_epoch = 0
best_parameters = None
train_history = []
validation_history = []

for epoch in range(10000):
    z1, cache1 = linear_forward(X_train, W1, b1)
    hidden, cache_relu = relu_forward(z1)
    predictions, cache2 = linear_forward(hidden, W2, b2)

    train_loss = mse_loss(predictions, y_train)
    dz2 = mse_backward(predictions, y_train)
    d_hidden, dW2, db2 = linear_backward(dz2, cache2)
    dz1 = relu_backward(d_hidden, cache_relu)
    dX, dW1, db1 = linear_backward(dz1, cache1)

    W1 -= alpha * dW1
    b1 -= alpha * db1
    W2 -= alpha * dW2
    b2 -= alpha * db2

    validation_z1, _ = linear_forward(X_validation, W1, b1)
    validation_hidden, _ = relu_forward(validation_z1)
    validation_predictions, _ = linear_forward(validation_hidden, W2, b2)
    validation_loss = mse_loss(validation_predictions, y_validation)

    train_history.append(train_loss)
    validation_history.append(validation_loss)

    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        best_epoch = epoch
        best_parameters = (W1.copy(), b1.copy(), W2.copy(), b2.copy())
        wait = 0
    else:
        wait += 1

    if wait >= patience:
        print('early stopping at epoch =', epoch)
        break

W1, b1, W2, b2 = best_parameters
print('best_epoch =', best_epoch)
print('best_validation_loss =', best_validation_loss)


运行后检查三个问题：

1. 保存的 `best_epoch` 是否一定是最后一个 epoch？
2. 为什么保存参数时必须 `.copy()`？
3. 如果验证误差一直下降，提前停止是否会立刻触发？

## 画出训练曲线

数字可以告诉我们最低点在哪里，曲线则能帮助我们看出误差变化的过程。竖线标出保存最佳参数的 epoch。

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(train_history, label='train loss')
plt.plot(validation_history, label='validation loss')
plt.axvline(best_epoch, color='red', linestyle='--', label='best epoch')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.show()


## Dropout：训练时随机关闭部分节点

Dropout 在训练时随机把一部分激活值置为 0，迫使网络不要过度依赖某几个节点。

这里使用 inverted dropout：保留下来的激活除以 `keep_prob`，使得训练时激活值的期望大小大致不变。

```python
mask = (random_values < keep_prob) / keep_prob
dropped = activations * mask
```

验证和测试时不使用随机掩码，直接保留全部节点。

In [ ]:
def dropout_forward(activations, keep_prob, rng, training=True):
    if not training:
        return activations, None

    mask = (rng.random(activations.shape) < keep_prob) / keep_prob
    return activations * mask, mask


def dropout_backward(dropped_gradient, mask):
    return dropped_gradient * mask


dropout_rng = np.random.default_rng(2)
activations = np.ones((2, 5))
dropped, mask = dropout_forward(activations, keep_prob=0.8, rng=dropout_rng)

print('mask =\n', mask)
print('dropped =\n', dropped)
print('dropped.shape =', dropped.shape)


观察这个示例：

- `mask` 的形状和激活值相同；
- 被关闭的位置是 `0`；
- 被保留的位置是 `1 / keep_prob`，这里是 `1.25`；
- 反向传播使用同一个 `mask`，所以被关闭的节点也不会收到梯度。

下一步把这个 Dropout 层插入训练网络，并比较训练模式与验证模式。

## 哪一句决定 Dropout 的节点？

真正决定哪些节点保留的是：

```python
random_values = rng.random(activations.shape)
keep = random_values < keep_prob
```

例如随机数为 `[0.2, 0.91, 0.6]`，且 `keep_prob=0.8`：

```text
random_values < 0.8 → [True, False, True]
```

因此第 1、3 个节点保留，第 2 个节点关闭。之后的：

```python
mask = keep / keep_prob
```

只负责把保留节点缩放为 `1 / keep_prob`，不负责选择节点。

`default_rng(2)` 中的 `2` 是随机种子。固定种子可以让每次运行生成相同的随机序列，方便调试和复现实验。

## 把 Dropout 插入训练流程

训练时，Dropout 放在隐藏层激活之后：

```text
linear → ReLU → Dropout → linear → loss
```

反向传播时顺序相反，先用同一个 `mask` 穿过 Dropout，再穿过 ReLU。验证集只经过 `linear → ReLU → linear`，不使用随机掩码。

In [ ]:
model_rng = np.random.default_rng(1)
dropout_rng = np.random.default_rng(2)
W1 = model_rng.normal(0, 0.5, size=(1, 32))
b1 = np.full(32, 0.1)
W2 = model_rng.normal(0, 0.5, size=(32, 1))
b2 = np.zeros(1)

alpha = 0.02
keep_prob = 0.8
dropout_train_history = []
dropout_validation_history = []

for epoch in range(2000):
    # 训练模式：使用 Dropout
    z1, cache1 = linear_forward(X_train, W1, b1)
    hidden, cache_relu = relu_forward(z1)
    hidden_dropout, dropout_mask = dropout_forward(
        hidden, keep_prob, dropout_rng, training=True
    )
    predictions, cache2 = linear_forward(hidden_dropout, W2, b2)

    train_loss = mse_loss(predictions, y_train)
    dz2 = mse_backward(predictions, y_train)
    d_hidden_dropout, dW2, db2 = linear_backward(dz2, cache2)
    d_hidden = dropout_backward(d_hidden_dropout, dropout_mask)
    dz1 = relu_backward(d_hidden, cache_relu)
    dX, dW1, db1 = linear_backward(dz1, cache1)

    W1 -= alpha * dW1
    b1 -= alpha * db1
    W2 -= alpha * dW2
    b2 -= alpha * db2

    # 验证模式：不使用 Dropout
    validation_z1, _ = linear_forward(X_validation, W1, b1)
    validation_hidden, _ = relu_forward(validation_z1)
    validation_predictions, _ = linear_forward(
        validation_hidden, W2, b2
    )
    validation_loss = mse_loss(validation_predictions, y_validation)

    dropout_train_history.append(train_loss)
    dropout_validation_history.append(validation_loss)

    if epoch % 500 == 0:
        print(
            'epoch =', epoch,
            'train_loss =', train_loss,
            'validation_loss =', validation_loss
        )


Dropout 会让训练误差出现波动，因为每一轮看到的子网络不同；这并不一定表示训练失败。更重要的是观察验证误差是否更稳定，以及最终泛化表现是否改善。